# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malaikasaleem944/malaika_flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
from pathlib import Path

# Load the public-safe starter dataset
DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")

Dataset shape: (30000, 44)
Loaded successfully.


In [4]:
feature_cols = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "avg_position",
    "ctr",
    "trend_pct",
]

X = df[feature_cols].copy()

for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.fillna(X.median())

print("Feature matrix shape:", X.shape)
print("Features used:")
print(feature_cols)
print("\nMissing values after filling:")
print(X.isna().sum())

Feature matrix shape: (30000, 6)
Features used:
['impressions_90d', 'sessions_90d', 'content_age_days', 'avg_position', 'ctr', 'trend_pct']

Missing values after filling:
impressions_90d     0
sessions_90d        0
content_age_days    0
avg_position        0
ctr                 0
trend_pct           0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "meaning": [
        "90-day impressions",
        "90-day sessions",
        "Age of content in days",
        "Average search position",
        "Click-through rate",
        "Percentage trend signal"
    ],
    "missing_after_fill": [
        X["impressions_90d"].isna().sum(),
        X["sessions_90d"].isna().sum(),
        X["content_age_days"].isna().sum(),
        X["avg_position"].isna().sum(),
        X["ctr"].isna().sum(),
        X["trend_pct"].isna().sum()
    ]
})

print(feature_notes.to_string(index=False))

         feature                 meaning  missing_after_fill
 impressions_90d      90-day impressions                   0
    sessions_90d         90-day sessions                   0
content_age_days  Age of content in days                   0
    avg_position Average search position                   0
             ctr      Click-through rate                   0
       trend_pct Percentage trend signal                   0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
# Look for columns that may be derived from the target or represent labels.
leakage_candidates = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "label",
        "target",
        "trend_direction"
    ])
]

print("Potential leakage/label-derived columns:")
print(leakage_candidates)

excluded_from_model = [
    col for col in leakage_candidates
    if col not in feature_cols
]

print("\nExcluded from model:")
print(excluded_from_model)

Potential leakage/label-derived columns:
['trend_direction']

Excluded from model:
['trend_direction']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [9]:
excluded_fields = {
    "trend_direction": "Label-derived categorical trend field; excluded to reduce leakage risk.",
    "is_declining_label": "Label-derived outcome field; excluded because it directly represents the target condition.",
    "client_name": "Not needed for prediction and excluded from the public-safe analysis.",
    "url": "Potentially identifying/private information; excluded from the public-facing model."
}

print("Excluded fields and reasons:")
for field, reason in excluded_fields.items():
    if field in df.columns:
        print(f"- {field}: {reason}")

Excluded fields and reasons:
- trend_direction: Label-derived categorical trend field; excluded to reduce leakage risk.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.